# LSTM Autoencoder

## Load data

Set directory

In [2]:
import sys
import os

# Find the project root (Speciale_Kode)
current_dir = os.getcwd()
project_root = current_dir

# Looks for "Speciale_Kode" folder:
while os.path.basename(project_root) != "Speciale_Kode":
    project_root = os.path.dirname(project_root)

# Add to Python path
if project_root not in sys.path:
    sys.path.append(project_root)

Load data

In [3]:
import pandas as pd
from pathlib import Path
from Modules.read_data import read_data

PRICE_ZONE = "DK1"  # "DK1" or "DK2"
TRAIN_WINDOW = 2 * 8760
VAL_START = "2024-01-01 00:00:00"
VAL_WINDOW = 8784
PREDICT_PERIOD = 4 * 168
STRIDE = 13 * 168                           # Stride is measured from the start of the previous fold.
POST_VALIDATION_EXCLUDE_HOURS = 168         # Exclude first 168h after each validation window from remainder_2024_for_train
INCLUDE_REMAINING_2024_DURING_TRAINING = True
# DKPrice is handled internally by the encoder; it is NOT added as a feature column here.
INCLUDE_PRICE_HISTORY_AS_INPUT = False      # True -> incldues DKPrice as input.
INCLUDE_PRICE_LAG1_AS_INPUT = True        # True -> adds DKPrice_lag1 as an input feature regardless of INCLUDE_LAGS
INCLUDE_LAGS = False                        # Include lag features (Price_lag1, Price_lag24, etc.) in training input
USE_FORECASTED_HISTORY = True               # Use model-predicted prices to compute lag features during prediction

(
    DK1_train,
    DK1_test,
    DK2_train,
    DK2_test,
    DK1_train_weather,
    DK1_test_weather,
    DK2_train_weather,
    DK2_test_weather
) = read_data("combined_data_cleaned_v5.csv")

if PRICE_ZONE == "DK1":
    dataset_train = DK1_train.copy()
    dataset_test = DK1_test.copy()
elif PRICE_ZONE == "DK2":
    dataset_train = DK2_train.copy()
    dataset_test = DK2_test.copy()
else:
    raise ValueError("PRICE_ZONE must be 'DK1' or 'DK2'.")

# read_data already returns all of 2024 in dataset_train and all of 2025 in dataset_test.
# Build the custom 2024 rolling validation split entirely from dataset_train.
dataset_train = dataset_train.sort_values("Time").reset_index(drop=True)
dataset_test = dataset_test.sort_values("Time").reset_index(drop=True)

val_start_ts = pd.Timestamp(VAL_START)
year_2024_start = pd.Timestamp("2024-01-01 00:00:00")
year_2025_start = pd.Timestamp("2025-01-01 00:00:00")

# Keep legacy full timeline variable before redefining dataset_train below.
df = pd.concat([dataset_train, dataset_test], ignore_index=True).sort_values("Time").reset_index(drop=True)

# Fixed history block: TRAIN_WINDOW ending at VAL_START.
history = dataset_train.loc[dataset_train["Time"] < val_start_ts].copy().iloc[-TRAIN_WINDOW:]
if len(history) < TRAIN_WINDOW:
    raise ValueError(
        f"Not enough history for TRAIN_WINDOW={TRAIN_WINDOW}. Got {len(history)} rows before {VAL_START}."
    )

data_2024 = dataset_train.loc[
    (dataset_train["Time"] >= year_2024_start) & (dataset_train["Time"] < year_2025_start)
].copy()

validation_idx = []
validation_windows = []
window_start = val_start_ts

# Validation windows in 2024: next fold starts STRIDE hours after the current fold start.
# Only full windows are allowed; trailing partial windows are skipped.
while (window_start + pd.Timedelta(hours=PREDICT_PERIOD)) <= year_2025_start:
    window_end = window_start + pd.Timedelta(hours=PREDICT_PERIOD)
    if window_end <= window_start:
        break

    mask = (data_2024["Time"] >= window_start) & (data_2024["Time"] < window_end)
    if mask.any():
        validation_idx.extend(data_2024.index[mask].tolist())
        validation_windows.append((window_start, window_end))

    window_start = window_start + pd.Timedelta(hours=STRIDE)

validation_idx = sorted(set(validation_idx))

# Exclude first POST_VALIDATION_EXCLUDE_HOURS after each validation window from train remainder.
post_validation_exclusion_idx = []
for _, window_end in validation_windows:
    exclusion_end = min(window_end + pd.Timedelta(hours=POST_VALIDATION_EXCLUDE_HOURS), year_2025_start)
    if exclusion_end <= window_end:
        continue

    exclusion_mask = (data_2024["Time"] >= window_end) & (data_2024["Time"] < exclusion_end)
    if exclusion_mask.any():
        post_validation_exclusion_idx.extend(data_2024.index[exclusion_mask].tolist())

post_validation_exclusion_idx = sorted(set(post_validation_exclusion_idx))
excluded_from_remainder_idx = sorted(set(validation_idx).union(post_validation_exclusion_idx))

dataset_validation = data_2024.loc[validation_idx].copy().sort_values("Time").reset_index(drop=True)
remainder_2024_for_train = data_2024.drop(index=excluded_from_remainder_idx).copy().sort_values("Time").reset_index(drop=True)

# Load cell is the only place that decides whether 2024 remainder is included in training.
if INCLUDE_REMAINING_2024_DURING_TRAINING:
    dataset_train = (
        pd.concat([history, remainder_2024_for_train], ignore_index=True)
        .sort_values("Time")
        .drop_duplicates(subset=["Time"], keep="last")
        .reset_index(drop=True)
    )
else:
    dataset_train = history.copy().sort_values("Time").reset_index(drop=True)

# Full context dataset: pre-2024 history + all of 2024.
# Used by get_predictions for lag computation regardless of training flags.
dataset_context = (
    pd.concat([history, data_2024], ignore_index=True)
    .sort_values("Time")
    .drop_duplicates(subset=["Time"], keep="last")
    .reset_index(drop=True)
)

# Preserve full target-bearing datasets for training/evaluation and create input views for later cells.
dataset_train_full = dataset_train.copy()
dataset_validation_full = dataset_validation.copy()
dataset_train_input = dataset_train_full.copy()
dataset_validation_input = dataset_validation_full.copy()

if not INCLUDE_PRICE_HISTORY_AS_INPUT:
    dataset_train_input = dataset_train_input.drop(columns=["DKPrice"])
    dataset_validation_input = dataset_validation_input.drop(columns=["DKPrice"])

lag_columns = [c for c in dataset_train_input.columns if '_lag' in c]
if not INCLUDE_LAGS:
    dataset_train_input = dataset_train_input.drop(columns=lag_columns, errors='ignore')
    dataset_validation_input = dataset_validation_input.drop(
        columns=[c for c in dataset_validation_input.columns if '_lag' in c], errors='ignore'
    )

if INCLUDE_PRICE_LAG1_AS_INPUT:
    dataset_train_input["DKPrice_lag1"] = dataset_train_full["Price_lag1"].values
    dataset_validation_input["DKPrice_lag1"] = dataset_validation_full["Price_lag1"].values

# Keep 2025 as test set.
dataset_test = dataset_test.copy().reset_index(drop=True)

target_time = val_start_ts
prices = history["DKPrice"].astype(float).values.reshape(-1, 1)

def _load_feature_predictions_for_zone(zone):
    prediction_path = Path(project_root) / "Data" / f"feature_predictions_{zone}_2024-2025.csv"
    if not prediction_path.exists():
        print("No precomputed forecasts found.")
        return None

    predictions = pd.read_csv(prediction_path, sep=";", decimal=".", parse_dates=["Time"], dayfirst=True)
    predictions = predictions.loc[:, ~predictions.columns.duplicated()].copy()
    if "DKZone" in predictions.columns:
        predictions = predictions.loc[predictions["DKZone"] == zone].copy()
        print(f"Loaded {len(predictions)} forecasts for zone {zone}.")
        print(f"Forecast features: {len(predictions.columns)} {predictions.columns.tolist()}")
    return predictions

print(f"Using zone: {PRICE_ZONE}")
print(f"Train source shape (all of 2024): {DK1_train.shape if PRICE_ZONE == 'DK1' else DK2_train.shape}")
print(f"Test source shape (all of 2025): {DK1_test.shape if PRICE_ZONE == 'DK1' else DK2_test.shape}")
print(f"Include remainder_2024_for_train in training: {INCLUDE_REMAINING_2024_DURING_TRAINING}")
print(f"Include DKPrice in training input dataset: {INCLUDE_PRICE_HISTORY_AS_INPUT}")
print(f"Include lag features in training input: {INCLUDE_LAGS}")
print(f"Include DKPrice_lag1 as input: {INCLUDE_PRICE_LAG1_AS_INPUT}")
print(f"Use forecasted prices for lag computation during prediction: {USE_FORECASTED_HISTORY}")
print(f"Train shape (prepared training dataset): {dataset_train.shape}")
print(f"Training input shape: {dataset_train_input.shape}")
print(f"2024 remainder rows included in training: {len(remainder_2024_for_train) if INCLUDE_REMAINING_2024_DURING_TRAINING else 0}")
print(f"Validation shape (rolling 2024 windows): {dataset_validation.shape}")
print(f"Validation input shape: {dataset_validation_input.shape}")
print(f"Context shape (history + all 2024): {dataset_context.shape}")
print(f"Test shape (2025): {dataset_test.shape}")
print(f"Validation windows created: {len(validation_windows)}")
print(f"Post-validation exclusion hours: {POST_VALIDATION_EXCLUDE_HOURS}")
print(f"Rows excluded from remainder after validation windows: {len(post_validation_exclusion_idx)}")
if validation_windows:
    print("All validation windows:")
    for idx, (window_start, window_end) in enumerate(validation_windows, start=1):
        print(f"  {idx:02d}. {window_start} -> {window_end}")
print(f"Training dataset columns: {dataset_train.columns.tolist()}")
print(f"Training input columns: {dataset_train_input.columns.tolist()}")

feature_predictions = _load_feature_predictions_for_zone(PRICE_ZONE)    
if feature_predictions is not None:
    print("\nPrecomputed forecasts loaded.")

use_precomputed_feature_values = feature_predictions is not None


Notebook_dir: c:\Users\n_and\OneDrive\Delt skrivebord\Data Science\Speciale\Energinet\Delte scripts\Speciale_Kode\Modules
Python_dir: c:\Users\n_and\OneDrive\Delt skrivebord\Data Science\Speciale\Energinet\Delte scripts\Speciale_Kode
Data_folder: c:\Users\n_and\OneDrive\Delt skrivebord\Data Science\Speciale\Energinet\Delte scripts\Speciale_Kode\Data
Training data shape (DK1): (78888, 38)
Test data shape (DK1): (8760, 38)
Test set fraction (DK1): 9.99%
Training data shape (DK2): (78888, 38)
Test data shape (DK2): (8760, 38)
Test set fraction (DK2): 9.99%
Using zone: DK1
Train source shape (all of 2024): (78888, 38)
Test source shape (all of 2025): (8760, 38)
Include remainder_2024_for_train in training: True
Include DKPrice in training input dataset: False
Include lag features in training input: False
Include DKPrice_lag1 as input: True
Use forecasted prices for lag computation during prediction: True
Train shape (prepared training dataset): (22944, 38)
Training input shape: (22944, 34)

Load Random Forest forecasting models

In [4]:
from Modules.Load_RF_forecast_models import load_rf_models

rf_models = None
if not use_precomputed_feature_values:
    # load_rf_models currently supports only the optional timeout argument.
    rf_models = load_rf_models(user="Nikolaj")      # set user to "Nikolaj" or "Christine"

Test CUDA

In [4]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA DIAGNOSTICS")
print("\nBasic Info:")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"\nGPU Info:")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
    print(f"Device Count: {torch.cuda.device_count()}")

    test_tensor = torch.randn(100, 100).to(device)
    print(f"Tensor on CUDA: {test_tensor.is_cuda}")

else:
    print("\n  Running on CPU - no CUDA available")

CUDA DIAGNOSTICS

Basic Info:
CUDA available: True
Device: cuda

GPU Info:
GPU Name: NVIDIA GeForce RTX 5060 Ti
CUDA Version: 12.8
cuDNN Version: 91002
Device Count: 1
Tensor on CUDA: True


### Helper functions

In [5]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.base import BaseEstimator, RegressorMixin
from torch.utils.data import DataLoader, Dataset


def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def smape_mean(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.abs(y_true) + np.abs(y_pred)
    vals = np.where(denom == 0, 0.0, 200.0 * np.abs(y_pred - y_true) / denom)
    return float(np.mean(vals))


# ---------------------------------------------------------------------------
# Seq2Seq Dataset
# ---------------------------------------------------------------------------

class Seq2SeqDataset(Dataset):
    """
    Builds encoder/decoder/target triplets on-the-fly for seq2seq training.

    For each valid index i in [seq_len, n_samples - horizon):
      encoder_input  = concat(X[i-seq_len:i], y[i-seq_len:i].reshape(-1,1))
                       shape (seq_len, n_features + 1)  – scaled features + unscaled DKPrice
      decoder_input  = X[i : i+horizon]
                       shape (horizon, n_features)       – scaled future features
      target         = y[i : i+horizon]
                       shape (horizon,)                  – future DKPrice values
    """

    def __init__(self, X_np: np.ndarray, y_np: np.ndarray, seq_len: int, horizon: int):
        self.X = X_np
        self.y = y_np
        self.seq_len = seq_len
        self.horizon = horizon
        self.n_valid = max(0, len(X_np) - seq_len - horizon + 1)

    def __len__(self) -> int:
        return self.n_valid

    def __getitem__(self, idx: int):
        i = idx + self.seq_len
        enc_x = self.X[i - self.seq_len : i]                          # (seq_len, n_features)
        enc_price = self.y[i - self.seq_len : i].reshape(-1, 1)       # (seq_len, 1)
        encoder_input = np.concatenate([enc_x, enc_price], axis=1)    # (seq_len, n_features+1)
        decoder_input = self.X[i : i + self.horizon]                  # (horizon, n_features)
        target = self.y[i : i + self.horizon]                         # (horizon,)
        return (
            torch.tensor(encoder_input, dtype=torch.float32),
            torch.tensor(decoder_input, dtype=torch.float32),
            torch.tensor(target, dtype=torch.float32),
        )


# ---------------------------------------------------------------------------
# LSTM Autoencoder module
# ---------------------------------------------------------------------------

class LSTMAutoencoder(nn.Module):
    """
    Seq2seq LSTM Autoencoder for multi-step price forecasting.

    Architecture (based on Option 1 in the design document):
      1. Feature encoder  – a small dense MLP applied per timestep that
                            compresses (n_features + 1) inputs to latent_dim.
      2. Encoder LSTM     – processes the latent sequence and produces a
                            compressed hidden state.
      3. Decoder LSTM     – initialised with the encoder's final hidden state
                            and fed future feature forecasts; outputs
                            decoder_horizon price predictions.

    Parameters
    ----------
    encoder_input_size : int
        Number of encoder input features per timestep (n_features + 1 for DKPrice).
    decoder_input_size : int
        Number of decoder input features per timestep (n_features, no DKPrice).
    latent_dim : int
        Output dimension of the per-timestep feature encoder.
    encoder_hidden_size : int
        Hidden size of the encoder LSTM.
    decoder_hidden_size : int
        Hidden size of the decoder LSTM.
    layers : int
        Number of LSTM layers (shared between encoder and decoder).
    dense_layers : int
        Depth of the per-timestep feature encoder MLP (>=1).
    dropout : float
        Dropout applied between LSTM layers (only active when layers > 1).
    """

    def __init__(
        self,
        encoder_input_size: int,
        decoder_input_size: int,
        latent_dim: int = 16,
        encoder_hidden_size: int = 64,
        decoder_hidden_size: int = 64,
        layers: int = 1,
        dense_layers: int = 1,
        dropout: float = 0.0,
    ):
        super().__init__()

        # --- Feature encoder (dense MLP applied per timestep) ---
        enc_layers = []
        in_size = encoder_input_size
        for _ in range(dense_layers - 1):
            enc_layers.append(nn.Linear(in_size, latent_dim))
            enc_layers.append(nn.ReLU())
            if dropout > 0.0:
                enc_layers.append(nn.Dropout(dropout))
            in_size = latent_dim
        enc_layers.append(nn.Linear(in_size, latent_dim))
        self.feature_encoder = nn.Sequential(*enc_layers)

        lstm_dropout = dropout if layers > 1 else 0.0

        # --- Encoder LSTM ---
        self.encoder_lstm = nn.LSTM(
            input_size=latent_dim,
            hidden_size=encoder_hidden_size,
            num_layers=layers,
            dropout=lstm_dropout,
            batch_first=True,
        )

        # --- Optional hidden-state adapter ---
        self.needs_adapter = (encoder_hidden_size != decoder_hidden_size)
        if self.needs_adapter:
            self.hidden_adapter = nn.Linear(encoder_hidden_size, decoder_hidden_size)

        # --- Decoder LSTM ---
        self.decoder_lstm = nn.LSTM(
            input_size=decoder_input_size,
            hidden_size=decoder_hidden_size,
            num_layers=layers,
            dropout=lstm_dropout,
            batch_first=True,
        )

        # --- Output projection ---
        self.fc = nn.Linear(decoder_hidden_size, 1)

    def forward(self, encoder_x: torch.Tensor, decoder_x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        encoder_x : Tensor of shape (batch, seq_len, encoder_input_size)
        decoder_x : Tensor of shape (batch, horizon, decoder_input_size)

        Returns
        -------
        Tensor of shape (batch, horizon)
        """
        # Feature encoding applied identically to every timestep
        latent = self.feature_encoder(encoder_x)          # (batch, seq_len, latent_dim)

        # Encoder LSTM – only final hidden state is used
        _, (h_enc, c_enc) = self.encoder_lstm(latent)     # (layers, batch, enc_hidden)

        # Adapt hidden state dimensions if encoder/decoder sizes differ
        if self.needs_adapter:
            h_dec = self.hidden_adapter(h_enc)
            c_dec = self.hidden_adapter(c_enc)
        else:
            h_dec, c_dec = h_enc, c_enc

        # Decoder LSTM initialized with encoder's final state
        out, _ = self.decoder_lstm(decoder_x, (h_dec, c_dec))  # (batch, horizon, dec_hidden)

        return self.fc(out).squeeze(-1)                    # (batch, horizon)


# ---------------------------------------------------------------------------
# Scikit-learn compatible regressor
# ---------------------------------------------------------------------------

class TorchLSTMAERegressor(BaseEstimator, RegressorMixin):
    """
    Scikit-learn style regressor wrapping LSTMAutoencoder.

    fit(X, y)
        X : (n_samples, n_features) – pre-scaled feature matrix (DKPrice excluded).
        y : (n_samples,)            – raw DKPrice target values.
        Internally builds Seq2SeqDataset and trains the seq2seq model.

    predict_ae(encoder_x, decoder_x)
        Used by week_predictions2_AE.get_predictions() for block inference.
        encoder_x : (1, seq_len, n_features+1) – scaled features + unscaled DKPrice
        decoder_x : (1, horizon, n_features)   – scaled future features
        Returns   : (horizon,) predicted prices.

    predict(X)
        2D fallback interface for SHAP / sklearn compatibility.
        Each row of X is broadcast to a constant 168-step decoder sequence;
        a zero encoder input is used as neutral baseline.
        Returns the mean prediction across the horizon.
    """

    def __init__(
        self,
        latent_dim: int = 16,
        encoder_hidden_size: int = 64,
        decoder_hidden_size: int = 64,
        layers: int = 1,
        dense_layers: int = 1,
        learning_rate: float = 1e-3,
        epochs: int = 40,
        batch_size: int = 32,
        sequence_length: int = 168,
        decoder_horizon: int = 168,
        dropout: float = 0.0,
        random_state: int = 42,
        log_epoch_metrics: bool = False,
        log_prefix: str = "",
        warm_start: bool = False,
    ):
        self.latent_dim = latent_dim
        self.encoder_hidden_size = encoder_hidden_size
        self.decoder_hidden_size = decoder_hidden_size
        self.layers = layers
        self.dense_layers = dense_layers
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.sequence_length = sequence_length
        self.decoder_horizon = decoder_horizon
        self.dropout = dropout
        self.random_state = random_state
        self.log_epoch_metrics = log_epoch_metrics
        self.log_prefix = log_prefix
        self.warm_start = warm_start

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _initialize_model_state(self, encoder_input_size: int, decoder_input_size: int):
        self.device_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.pin_memory_ = self.device_.type == "cuda"
        self.encoder_input_size_ = int(encoder_input_size)
        self.decoder_input_size_ = int(decoder_input_size)
        self.model_ = LSTMAutoencoder(
            encoder_input_size=self.encoder_input_size_,
            decoder_input_size=self.decoder_input_size_,
            latent_dim=int(self.latent_dim),
            encoder_hidden_size=int(self.encoder_hidden_size),
            decoder_hidden_size=int(self.decoder_hidden_size),
            layers=int(self.layers),
            dense_layers=int(self.dense_layers),
            dropout=float(self.dropout),
        ).to(self.device_)

        # Load pretrained feature encoder weights from Stage 1 (if set via
        # set_pretrained_encoder()).  Encoder params are frozen when freeze=True,
        # so the optimizer below excludes them automatically.
        _sd = getattr(self, "_pretrained_encoder_state_dict_", None)
        if _sd is not None:
            self.model_.feature_encoder.load_state_dict(_sd)
            if getattr(self, "_freeze_encoder_", True):
                for p in self.model_.feature_encoder.parameters():
                    p.requires_grad = False

        self.loss_fn_ = nn.MSELoss()
        # Only pass parameters that require gradients so frozen encoder layers
        # are not updated even if the optimizer sees them.
        self.optimizer_ = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.model_.parameters()),
            lr=float(self.learning_rate),
        )
        self.epoch_losses_ = []
        self.epoch_smapes_ = []
        self.epoch_maes_ = []
        self.epoch_rmses_ = []
        self._epochs_trained_ = 0

    # ------------------------------------------------------------------
    # set_pretrained_encoder
    # ------------------------------------------------------------------

    def set_pretrained_encoder(
        self, state_dict: dict, freeze: bool = True
    ) -> "TorchLSTMAERegressor":
        """
        Store pretrained feature encoder weights to be loaded at the next model
        initialisation (i.e. the first fit() call, or when warm_start=False).

        Call this before the first fit() / run_cross_validation() call so the
        weights are in place when _initialize_model_state() creates model_.

        Parameters
        ----------
        state_dict : OrderedDict returned by FeatureAutoencoder.encoder.state_dict()
                     (i.e. ``result["best_encoder_state_dict"]`` from Stage 1).
        freeze     : if True the feature_encoder parameters are frozen so only
                     the encoder/decoder LSTM and output layers are updated
                     during Stage 2 training.
        """
        import copy as _copy
        self._pretrained_encoder_state_dict_ = _copy.deepcopy(state_dict)
        self._freeze_encoder_ = freeze
        return self

    # ------------------------------------------------------------------
    # fit
    # ------------------------------------------------------------------

    def fit(self, X, y):
        set_seed(self.random_state)

        X_np = np.asarray(X, dtype=np.float32)
        y_np = np.asarray(y, dtype=np.float32).reshape(-1)

        if X_np.ndim != 2:
            raise ValueError(f"Expected 2-D X, got shape {X_np.shape}.")

        n_decoder_features = X_np.shape[1]
        n_encoder_features = n_decoder_features + 1   # features + DKPrice
        seq_len = int(self.sequence_length)
        horizon = int(self.decoder_horizon)

        # Build or reuse the dataset (reused across warm-start epochs for speed)
        needs_rebuild = (
            not hasattr(self, "_train_dataset_")
            or getattr(self, "_train_X_shape_", None) != X_np.shape
            or getattr(self, "_train_y_len_", None) != len(y_np)
        )
        if needs_rebuild:
            self._train_dataset_ = Seq2SeqDataset(X_np, y_np, seq_len, horizon)
            self._train_X_shape_ = X_np.shape
            self._train_y_len_ = len(y_np)

        if len(self._train_dataset_) == 0:
            raise ValueError(
                f"Training set too small to build any seq2seq sample "
                f"(need > seq_len+horizon={seq_len+horizon} rows, got {len(X_np)})."
            )

        # Initialise or reuse the model
        needs_reinit = (
            (not bool(self.warm_start))
            or (not hasattr(self, "model_"))
            or (not hasattr(self, "encoder_input_size_"))
            or (int(self.encoder_input_size_) != n_encoder_features)
            or (int(self.decoder_input_size_) != n_decoder_features)
        )
        if needs_reinit:
            self._initialize_model_state(
                encoder_input_size=n_encoder_features,
                decoder_input_size=n_decoder_features,
            )
        elif not hasattr(self, "device_"):
            self.device_ = next(self.model_.parameters()).device
            self.pin_memory_ = self.device_.type == "cuda"

        loader = DataLoader(
            self._train_dataset_,
            batch_size=int(self.batch_size),
            shuffle=True,
            pin_memory=self.pin_memory_,
        )

        self.model_.train()
        for _ in range(int(self.epochs)):
            batch_losses = []
            epoch_preds = []
            epoch_targets = []

            for enc_batch, dec_batch, tgt_batch in loader:
                enc_batch = enc_batch.to(self.device_, non_blocking=self.pin_memory_)
                dec_batch = dec_batch.to(self.device_, non_blocking=self.pin_memory_)
                tgt_batch = tgt_batch.to(self.device_, non_blocking=self.pin_memory_)

                self.optimizer_.zero_grad()
                preds = self.model_(enc_batch, dec_batch)   # (batch, horizon)
                loss = self.loss_fn_(preds, tgt_batch)
                loss.backward()
                self.optimizer_.step()

                batch_losses.append(float(loss.item()))
                epoch_preds.append(preds.detach().cpu().numpy().reshape(-1))
                epoch_targets.append(tgt_batch.detach().cpu().numpy().reshape(-1))

            epoch_loss = float(np.mean(batch_losses)) if batch_losses else float("nan")
            self.epoch_losses_.append(epoch_loss)

            if epoch_preds and epoch_targets:
                y_pred_epoch = np.concatenate(epoch_preds)
                y_true_epoch = np.concatenate(epoch_targets)
                epoch_smape = smape_mean(y_true_epoch, y_pred_epoch)
                epoch_mae = float(np.mean(np.abs(y_true_epoch - y_pred_epoch)))
                epoch_rmse = float(np.sqrt(np.mean((y_true_epoch - y_pred_epoch) ** 2)))
            else:
                epoch_smape = epoch_mae = epoch_rmse = float("nan")

            self.epoch_smapes_.append(float(epoch_smape))
            self.epoch_maes_.append(float(epoch_mae))
            self.epoch_rmses_.append(float(epoch_rmse))
            self._epochs_trained_ += 1

            if bool(self.log_epoch_metrics):
                try:
                    import wandb
                    if wandb.run is not None:
                        pfx = self.log_prefix
                        wandb.log({
                            f"{pfx}train_MSE_loss" if pfx else "train_MSE_loss": epoch_loss,
                            f"{pfx}train_smape"    if pfx else "train_smape":    float(epoch_smape),
                            f"{pfx}train_mae"      if pfx else "train_mae":      float(epoch_mae),
                            f"{pfx}train_rmse"     if pfx else "train_rmse":     float(epoch_rmse),
                            f"{pfx}epoch"          if pfx else "epoch":          int(self._epochs_trained_),
                        })
                except Exception:
                    pass

        return self

    # ------------------------------------------------------------------
    # predict_ae  (primary inference path used by week_predictions2_AE)
    # ------------------------------------------------------------------

    def predict_ae(self, encoder_x: np.ndarray, decoder_x: np.ndarray) -> np.ndarray:
        """
        Predict one 168-hour block in a single forward pass.

        Parameters
        ----------
        encoder_x : ndarray of shape (1, seq_len, n_features+1)
            Scaled historical features concatenated with unscaled DKPrice.
        decoder_x : ndarray of shape (1, horizon, n_features)
            Scaled future feature forecasts.

        Returns
        -------
        ndarray of shape (horizon,)
        """
        self.model_.eval()
        enc_t = torch.tensor(encoder_x, dtype=torch.float32).to(self.device_)
        dec_t = torch.tensor(decoder_x, dtype=torch.float32).to(self.device_)
        with torch.no_grad():
            out = self.model_(enc_t, dec_t)    # (1, horizon)
        return out.squeeze(0).detach().cpu().numpy()

    # ------------------------------------------------------------------
    # predict  (2-D fallback for SHAP / sklearn compatibility)
    # ------------------------------------------------------------------

    def predict(self, X) -> np.ndarray:
        """
        2-D input interface for SHAP / sklearn compatibility.

        Each row of X is broadcast to a constant `decoder_horizon`-step decoder
        sequence.  A zero encoder input is used as a neutral baseline.
        Returns the mean prediction across the horizon for each sample.
        """
        X_np = np.asarray(X, dtype=np.float32)
        if X_np.ndim != 2:
            raise ValueError(
                f"predict() expects 2-D X; got shape {X_np.shape}. "
                "For block inference use predict_ae(encoder_x, decoder_x)."
            )

        n_samples, n_features = X_np.shape
        horizon = int(self.decoder_horizon)
        seq_len = int(self.sequence_length)

        # Neutral encoder (all zeros)
        enc_np = np.zeros((1, seq_len, n_features + 1), dtype=np.float32)
        enc_t = torch.tensor(enc_np, dtype=torch.float32).to(self.device_)

        self.model_.eval()
        preds = []
        with torch.no_grad():
            for i in range(n_samples):
                # Replicate single feature row across the full decoder horizon
                dec_np = np.tile(X_np[i : i + 1], (horizon, 1))[np.newaxis, :]  # (1, horizon, n)
                dec_t = torch.tensor(dec_np, dtype=torch.float32).to(self.device_)
                out = self.model_(enc_t, dec_t)   # (1, horizon)
                preds.append(float(out.mean().item()))

        return np.array(preds, dtype=np.float32)


## Hyperparameter search

### Step 1: Feature Encoder Search

Tune the per-timestep feature encoder MLP as a standalone autoencoder.  
Loss/evaluation metric: **SMAPE on reconstruction**.  
Optimal `latent_dim` and `dense_layers` are then carried forward into the full LSTM AE search.

In [6]:
import copy
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset


# ---------------------------------------------------------------------------
# Differentiable SMAPE loss
# ---------------------------------------------------------------------------

class SMAPELoss(nn.Module):
    """SMAPE loss: 200 * |y_pred - y_true| / (|y_true| + |y_pred| + eps)."""

    def __init__(self, eps: float = 1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        denom = torch.abs(y_true) + torch.abs(y_pred) + self.eps
        return torch.mean(200.0 * torch.abs(y_pred - y_true) / denom)


# ---------------------------------------------------------------------------
# Feature Autoencoder – mirrors the architecture of LSTMAutoencoder.feature_encoder
# ---------------------------------------------------------------------------

class FeatureAutoencoder(nn.Module):
    """
    Standalone autoencoder for tuning the per-timestep feature encoder MLP.

    Encoder architecture is identical to LSTMAutoencoder.feature_encoder:
      dense_layers=1 → single Linear(input_size → latent_dim)
      dense_layers=k → k-1 × [Linear → ReLU → (Dropout)] + final Linear → latent_dim

    Decoder mirrors the encoder (symmetric MLP from latent_dim back to input_size).

    Parameters
    ----------
    input_size  : number of features per timestep (n_features + 1 for DKPrice, same
                  as encoder_input_size in LSTMAutoencoder)
    latent_dim  : output dimension of the encoder (hyperparameter to tune)
    dense_layers: depth of the encoder/decoder MLP (≥1)
    dropout     : dropout rate applied between hidden layers (only if dense_layers > 1)
    """

    def __init__(
        self,
        input_size: int,
        latent_dim: int,
        dense_layers: int,
        dropout: float = 0.0,
    ):
        super().__init__()

        # ---- Encoder (same as LSTMAutoencoder.feature_encoder) ----
        enc = []
        in_sz = input_size
        for _ in range(dense_layers - 1):
            enc.append(nn.Linear(in_sz, latent_dim))
            enc.append(nn.ReLU())
            if dropout > 0.0:
                enc.append(nn.Dropout(dropout))
            in_sz = latent_dim
        enc.append(nn.Linear(in_sz, latent_dim))
        self.encoder = nn.Sequential(*enc)

        # ---- Decoder (symmetric) ----
        dec = []
        in_sz = latent_dim
        for _ in range(dense_layers - 1):
            dec.append(nn.Linear(in_sz, latent_dim))
            dec.append(nn.ReLU())
            if dropout > 0.0:
                dec.append(nn.Dropout(dropout))
            in_sz = latent_dim
        dec.append(nn.Linear(in_sz, input_size))
        self.decoder = nn.Sequential(*dec)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)


# ---------------------------------------------------------------------------
# Training helper
# ---------------------------------------------------------------------------

def train_feature_autoencoder(
    X_train: np.ndarray,
    X_val: np.ndarray,
    latent_dim: int,
    dense_layers: int,
    learning_rate: float,
    max_epochs: int,
    patience: int,
    batch_size: int,
    dropout: float = 0.0,
    random_state: int = 42,
    log_wandb: bool = False,
) -> dict:
    """
    Train a FeatureAutoencoder with SMAPE reconstruction loss.

    Parameters
    ----------
    X_train / X_val : float32 arrays of shape (n_samples, input_size).
                      Should be the scaled feature matrix concatenated with
                      raw DKPrice, matching the encoder input in Seq2SeqDataset.
    log_wandb       : whether to call wandb.log() per epoch.

    Returns
    -------
    dict with keys: best_val_smape, best_epoch, epochs_trained, epoch_history,
                    best_encoder_state_dict (state_dict of the encoder at best epoch,
                    ready to be loaded into LSTMAutoencoder.feature_encoder).
    """
    set_seed(random_state)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pin_memory = device.type == "cuda"

    input_size = X_train.shape[1]
    model = FeatureAutoencoder(
        input_size=input_size,
        latent_dim=latent_dim,
        dense_layers=dense_layers,
        dropout=dropout,
    ).to(device)

    criterion = SMAPELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    train_t = torch.tensor(X_train, dtype=torch.float32)
    val_t = torch.tensor(X_val, dtype=torch.float32)

    loader = DataLoader(
        TensorDataset(train_t),
        batch_size=batch_size,
        shuffle=True,
        pin_memory=pin_memory,
    )

    best_val_smape = float("inf")
    best_epoch = 0
    patience_counter = 0
    epoch_history = []
    epoch = 0
    best_encoder_state_dict = None   # saved at the epoch with lowest val SMAPE

    for epoch in range(1, max_epochs + 1):
        model.train()
        batch_losses = []
        for (x_batch,) in loader:
            x_batch = x_batch.to(device, non_blocking=pin_memory)
            optimizer.zero_grad()
            recon = model(x_batch)
            loss = criterion(recon, x_batch)
            loss.backward()
            optimizer.step()
            batch_losses.append(float(loss.item()))

        train_smape = float(np.mean(batch_losses)) if batch_losses else float("nan")

        model.eval()
        with torch.no_grad():
            val_recon = model(val_t.to(device))
            val_smape = float(criterion(val_recon, val_t.to(device)).item())

        epoch_history.append({"epoch": epoch, "train_smape": train_smape, "val_smape": val_smape})

        if log_wandb:
            try:
                import wandb as _wandb
                _wandb.log({"epoch": epoch, "train_smape": train_smape, "val_smape": val_smape})
            except Exception:
                pass

        if val_smape < best_val_smape:
            best_val_smape = val_smape
            best_epoch = epoch
            patience_counter = 0
            # Snapshot the encoder weights at this best epoch
            best_encoder_state_dict = copy.deepcopy(model.encoder.state_dict())
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return {
        "best_val_smape": best_val_smape,
        "best_epoch": best_epoch,
        "epochs_trained": epoch,
        "epoch_history": epoch_history,
        "best_encoder_state_dict": best_encoder_state_dict,
    }


Feature encoder search grid

In [7]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------------
# Hyperparameter grid for the feature encoder autoencoder search
# ---------------------------------------------------------------------------

fe_param_grid = {
    "latent_dim":    [16, 24, 33, 38, 48],
    "dense_layers":  [1, 2, 3],
    "learning_rate": [0.001, 0.0005],
    "batch_size":    [32, 64],
    "dropout":       [0.0, 0.2],   # expand to [0.0, 0.1] if dense_layers > 1 is explored with dropout
    "max_epochs":    [60],
    "patience":      [10],
}

fe_total_combinations = int(np.prod([len(v) for v in fe_param_grid.values()]))

# When dense_layers == 1, dropout has no effect; those combinations are redundant.
_other_keys = [k for k in fe_param_grid if k not in ("dense_layers", "dropout")]
_n_other = int(np.prod([len(fe_param_grid[k]) for k in _other_keys]))
_n_redundant = _n_other * 1 * len([d for d in fe_param_grid["dropout"] if d != 0.0])
fe_effective_combinations = fe_total_combinations - _n_redundant
print(f"Feature encoder total combinations: {fe_total_combinations}")
print(f"Combinations after skipping dropout>0 with 1 layer: {fe_effective_combinations}")

# ---------------------------------------------------------------------------
# Prepare input data: scaled features + raw DKPrice
# (Matches the encoder input built in Seq2SeqDataset)
# ---------------------------------------------------------------------------

fe_feature_cols = [c for c in dataset_train_input.columns if c not in ["Time", "DKPrice"]]
print(f"Feature columns ({len(fe_feature_cols)}): {fe_feature_cols}")

fe_X_features = dataset_train_input[fe_feature_cols].astype(np.float32).values
fe_y_price    = dataset_train_full["DKPrice"].astype(np.float32).values.reshape(-1, 1)

# Fit scaler only on the feature part (DKPrice is kept raw, matching Seq2SeqDataset behaviour)
fe_scaler = StandardScaler()
fe_X_scaled = fe_scaler.fit_transform(fe_X_features).astype(np.float32)

# Final input matrix: [scaled_features | raw_DKPrice]
fe_X = np.concatenate([fe_X_scaled, fe_y_price], axis=1)

# Chronological 90/10 split (no shuffling – temporal data)
fe_split_idx = int(len(fe_X) * 0.9)
fe_X_train = fe_X[:fe_split_idx]
fe_X_val   = fe_X[fe_split_idx:]

print(f"Train samples: {len(fe_X_train)},  val samples: {len(fe_X_val)}")
print(f"Feature AE input size: {fe_X.shape[1]}  (= {len(fe_feature_cols)} features + 1 DKPrice)")

Feature encoder total combinations: 120
Combinations after skipping dropout>0 with 1 layer: 100
Feature columns (32): ['OffshoreWindPower', 'OnshoreWindPower', 'HydroPower', 'SolarPower', 'Biomass', 'Biogas', 'Waste', 'FossilGas', 'FossilOil', 'FossilHardCoal', 'ExchangeGreatBelt', 'ExchangeGermany', 'ExchangeSweden', 'ExchangeNorway', 'ExchangeNetherlands', 'WindSpeed', 'Radiation', 'DEPrice', 'NO2Price', 'SE3Price', 'SE4Price', 'NLPrice', 'GrossCon', 'TotalProduction', 'Year', 'Month', 'Day', 'WeekDay', 'Hour', 'OffshoreWindCapacity', 'OnshoreWindCapacity', 'SolarPowerCapacity']
Train samples: 20649,  val samples: 2295
Feature AE input size: 33  (= 32 features + 1 DKPrice)


Run feature encoder search

In [8]:
import itertools
import wandb
import pandas as pd
from pathlib import Path
from time import time

FE_WANDB_PROJECT    = f"FeatureEncoder_AE_search_exclPriceLag1_exclPrice{PRICE_ZONE}"
FE_WANDB_RUN_BASE   = f"{PRICE_ZONE}_"
FE_START_COMBINATION = 1   # resume from a specific combination number if needed

fe_param_names  = list(fe_param_grid.keys())
fe_param_values = list(fe_param_grid.values())
_dense_idx   = fe_param_names.index("dense_layers")
_dropout_idx = fe_param_names.index("dropout")
fe_combinations = [
    combo for combo in itertools.product(*fe_param_values)
    if not (combo[_dense_idx] == 1 and combo[_dropout_idx] != 0.0)
]
fe_num_combinations = len(fe_combinations)
print(f"Running {fe_num_combinations} feature encoder combinations (skipped dropout>0 with dense_layers==1)")

fe_results  = []
fe_start_ts = time()

# Global best across all Stage 1 combinations – exposed to Stage 2.
best_stage1_val_smape         = float("inf")
best_stage1_encoder_state_dict = None   # encoder weights at best reconstruction SMAPE
best_stage1_params            = None    # hyperparameters of the best combination

for comb_no, combination in enumerate(fe_combinations, start=1):
    params = dict(zip(fe_param_names, combination))

    if comb_no < FE_START_COMBINATION:
        continue

    elapsed_min  = (time() - fe_start_ts) / 60
    est_total    = elapsed_min / comb_no * fe_num_combinations if comb_no > 1 else float("nan")
    print(
        f"\nFE combination {comb_no}/{fe_num_combinations}: {params}  "
        f"[{elapsed_min:.1f} min elapsed, ~{est_total:.1f} min total]"
    )

    run_name = (FE_WANDB_RUN_BASE +
        f"latentdim{params['latent_dim']}_dense{params['dense_layers']}"
        f"_lr{params['learning_rate']}_batchsize{params['batch_size']}"
        f"_dropout{params['dropout']}_comb{comb_no:03d}"
    )

    run = wandb.init(
        project=FE_WANDB_PROJECT,
        name=run_name,
        config={
            "price_zone":       PRICE_ZONE,
            "fe_input_size":    int(fe_X.shape[1]),
            "train_samples":    int(len(fe_X_train)),
            "val_samples":      int(len(fe_X_val)),
            "combination":      int(comb_no),
            "num_combinations": int(fe_num_combinations),
            **params,
        },
        tags=["feature-encoder", "autoencoder", "hyperparameter-search", "smape"],
        reinit=True,
        settings=wandb.Settings(start_method="thread"),
    )

    try:
        result = train_feature_autoencoder(
            X_train=fe_X_train,
            X_val=fe_X_val,
            latent_dim=int(params["latent_dim"]),
            dense_layers=int(params["dense_layers"]),
            learning_rate=float(params["learning_rate"]),
            max_epochs=int(params["max_epochs"]),
            patience=int(params["patience"]),
            batch_size=int(params["batch_size"]),
            dropout=float(params["dropout"]),
            log_wandb=True,
        )

        row = {
            **params,
            "combination":      int(comb_no),
            "fe_input_size":    int(fe_X.shape[1]),
            "price_zone":       PRICE_ZONE,
            "best_val_smape":   float(result["best_val_smape"]),
            "best_epoch":       int(result["best_epoch"]),
            "epochs_trained":   int(result["epochs_trained"]),
        }
        fe_results.append(row)

        run.summary.update({
            "best_val_smape": float(result["best_val_smape"]),
            "best_epoch":     int(result["best_epoch"]),
            "epochs_trained": int(result["epochs_trained"]),
        })

        print(
            f"  -> best val SMAPE: {result['best_val_smape']:.4f}  "
            f"(epoch {result['best_epoch']}/{result['epochs_trained']})"
        )

        # Track the globally best encoder weights across all Stage 1 combinations
        if result["best_val_smape"] < best_stage1_val_smape:
            best_stage1_val_smape          = float(result["best_val_smape"])
            best_stage1_encoder_state_dict = result["best_encoder_state_dict"]
            best_stage1_params             = dict(params)
            print(f"  *** New global best: val SMAPE={best_stage1_val_smape:.4f} ***")

    finally:
        wandb.finish()

# ---- Save & display results ----
fe_results_df = pd.DataFrame(fe_results).sort_values("best_val_smape").reset_index(drop=True)

output_folder = Path(project_root) / "Deep learners" / "LSTM Autoencoder"
output_folder.mkdir(parents=True, exist_ok=True)

fe_base_name = f"{PRICE_ZONE}_feature_encoder_search_results"
fe_out_path  = output_folder / f"{fe_base_name}.csv"
_counter = 1
while fe_out_path.exists():
    fe_out_path = output_folder / f"{fe_base_name}_{_counter}.csv"
    _counter += 1

# fe_results_df.to_csv(fe_out_path, index=False, decimal=",")
# print(f"\nFeature encoder search results saved to: {fe_out_path}")

print("\nTop 10 configurations by val SMAPE:")
display(fe_results_df.head(10))

# ---- Summary of what Stage 2 will receive ----
print(
    f"\n=== Stage 1 complete – best encoder weights ready for Stage 2 ===\n"
    f"  latent_dim           : {best_stage1_params['latent_dim']}\n"
    f"  dense_layers         : {best_stage1_params['dense_layers']}\n"
    f"  best_val_smape (recon): {best_stage1_val_smape:.4f}\n"
    f"\n  best_stage1_encoder_state_dict is available for Stage 2.\n"
    f"  These weights will be loaded into LSTMAutoencoder.feature_encoder\n"
    f"  and frozen during Stage 2 LSTM tuning.")

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


Running 100 feature encoder combinations (skipped dropout>0 with dense_layers==1)

FE combination 1/100: {'latent_dim': 16, 'dense_layers': 1, 'learning_rate': 0.001, 'batch_size': 32, 'dropout': 0.0, 'max_epochs': 60, 'patience': 10}  [0.0 min elapsed, ~nan min total]


wandb: Currently logged in as: nande24 (Energinet_speciale) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


  -> best val SMAPE: 45.5865  (epoch 41/51)
  *** New global best: val SMAPE=45.5865 ***


epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
train_smape,█▇▇▇▆▄▄▅▄▆▄▄▃▃▅▃▃▃▃▂▂▂▂▂▁▂▁▁▁▂▁▁▁▁▁▁▁▂▂▁
val_smape,█▇▆▆▅▅▄▄▅▄▆▄▄▄▃▂▂▂▂▂▃▂▂▁▂▂▁▂▂▁▁▂▁▁▁▂▂▂▁▁
best_epoch,41
best_val_smape,45.58654
epoch,51
epochs_trained,51
train_smape,46.2032
val_smape,50.48309


...

  -> best val SMAPE: 86.0543  (epoch 60/60)


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇██
train_smape,█▆▅▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁
val_smape,█▆▆▆▆▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▂▂▁▁
best_epoch,60
best_val_smape,86.05431
epoch,60
epochs_trained,60
train_smape,79.62434
val_smape,86.05431



Top 10 configurations by val SMAPE:


,latent_dim,dense_layers,learning_rate,batch_size,dropout,max_epochs,patience,combination,fe_input_size,price_zone,best_val_smape,best_epoch,epochs_trained
0,48,1,0.0010,64,0.0,60,10,82,33,DK1,23.741976,53,60
1,33,1,0.0010,32,0.0,60,10,41,33,DK1,25.050196,45,55
2,38,1,0.0010,32,0.0,60,10,61,33,DK1,32.206215,22,32
3,24,1,0.0010,32,0.0,60,10,21,33,DK1,32.499565,35,45
4,33,1,0.0010,64,0.0,60,10,42,33,DK1,37.741501,24,34
5,48,1,0.0010,32,0.0,60,10,81,33,DK1,42.390179,15,25
6,48,1,0.0005,32,0.0,60,10,83,33,DK1,42.804047,52,60
7,38,2,0.0005,32,0.0,60,10,69,33,DK1,45.561584,57,60
8,16,1,0.0010,32,0.0,60,10,1,33,DK1,45.586540,41,51
9,24,1,0.0010,64,0.0,60,10,22,33,DK1,46.392319,25,35



=== Stage 1 complete – best encoder weights ready for Stage 2 ===
  latent_dim           : 48
  dense_layers         : 1
  best_val_smape (recon): 23.7420

  best_stage1_encoder_state_dict is available for Stage 2.
  These weights will be loaded into LSTMAutoencoder.feature_encoder
  and frozen during Stage 2 LSTM tuning.


### Step 2: Full LSTM AE Search

Search grid

In [11]:
import numpy as np
import itertools

# ---------------------------------------------------------------------------
# Step 2 search grid – split by component, combined at the end.
# latent_dim and dense_layers are NOT swept here; they are fixed to the
# best values found in Stage 1 (best_stage1_params).
# ---------------------------------------------------------------------------

# 1. Encoder LSTM
enc_lstm_grid = {
    "encoder_hidden_size": [16, 32, 48],
    "layers":              [1, 2],
}

# 2. Decoder LSTM
dec_lstm_grid = {
    "decoder_hidden_size": [16, 32, 48],
}

# 3. Shared / training hyperparameters
training_grid = {
    "learning_rate":   [0.001],
    "max_epochs":      [60],
    "patience":        [10],
    "batch_size":      [32, 64],
    "sequence_length": [24, 168],
    "dropout":         [0.0, 0.2],
}

# Combine LSTM + training sub-grids into one flat grid
param_grid = {**enc_lstm_grid, **dec_lstm_grid, **training_grid}

total_combinations = int(np.prod([len(v) for v in param_grid.values()]))

# Exclude layers==1 with dropout>0 (dropout has no effect on a single-layer LSTM)
excluded = sum(
    1
    for combo in itertools.product(*param_grid.values())
    if dict(zip(param_grid.keys(), combo))["layers"] == 1
    and dict(zip(param_grid.keys(), combo))["dropout"] > 0.0
)
effective_combinations = total_combinations - excluded

print(f"Stage 1 encoder shape (fixed for Stage 2):")
if "best_stage1_params" in globals() and best_stage1_params is not None:
    print(f"  latent_dim   = {best_stage1_params['latent_dim']}")
    print(f"  dense_layers = {best_stage1_params['dense_layers']}")
else:
    print("  (run Stage 1 first to set best_stage1_params)")

print(f"\nSub-grid sizes:")
print(f"  Encoder LSTM     : {int(np.prod([len(v) for v in enc_lstm_grid.values()]))} combinations  {enc_lstm_grid}")
print(f"  Decoder LSTM     : {int(np.prod([len(v) for v in dec_lstm_grid.values()]))} combinations  {dec_lstm_grid}")
print(f"  Training / shared: {int(np.prod([len(v) for v in training_grid.values()]))} combinations  {training_grid}")
print(f"\nTotal combinations (before exclusions): {total_combinations}")
print(f"Excluded (layers=1 & dropout>0):        {excluded}")
print(f"Effective combinations:                 {effective_combinations}")


Stage 1 encoder shape (fixed for Stage 2):
  latent_dim   = 48
  dense_layers = 1

Sub-grid sizes:
  Encoder LSTM     : 6 combinations  {'encoder_hidden_size': [16, 32, 48], 'layers': [1, 2]}
  Decoder LSTM     : 3 combinations  {'decoder_hidden_size': [16, 32, 48]}
  Training / shared: 8 combinations  {'learning_rate': [0.001], 'max_epochs': [60], 'patience': [10], 'batch_size': [32, 64], 'sequence_length': [24, 168], 'dropout': [0.0, 0.2]}

Total combinations (before exclusions): 144
Excluded (layers=1 & dropout>0):        36
Effective combinations:                 108


Hyperparameter search

In [12]:

from Modules.Validation3_AE import run_cross_validation
import itertools
import numpy as np
from pathlib import Path
from time import time

import pandas as pd
import wandb

# ---------------------------------------------------------------------------
# Verify that Stage 1 produced a pretrained encoder state dict
# ---------------------------------------------------------------------------
if "best_stage1_encoder_state_dict" not in globals() or best_stage1_encoder_state_dict is None:
    raise RuntimeError(
        "best_stage1_encoder_state_dict is not set. Run the Stage 1 feature encoder "
        "search cell first so that pretrained encoder weights are available."
    )

print(
    f"Using pretrained encoder from Stage 1:\n"
    f"  latent_dim   = {best_stage1_params['latent_dim']}\n"
    f"  dense_layers = {best_stage1_params['dense_layers']}\n"
    f"  best recon SMAPE = {best_stage1_val_smape:.4f}\n"
)

PREDICT_PERIOD = 1 * 168
split_setup = 2
start_combination = 1

WANDB_PROJECT = "LSTM_AE_param_search_exclPrice_exclPriceLag1_DK1"

# param_grid is defined in the grid cell above (split into fe_grid, enc_lstm_grid,
# dec_lstm_grid and training_grid, then merged into one flat param_grid dict).
# Stage 2 fixes latent_dim and dense_layers to the Stage 1 best values; only
# LSTM / training hyperparameters are swept here.
num_combinations = int(np.prod([len(v) for v in param_grid.values()]))
print(f"Total combinations: {num_combinations}")

param_names = list(param_grid.keys())
param_values = list(param_grid.values())
all_combinations = list(itertools.product(*param_values))

# Feature columns: all columns except Time and DKPrice.
# DKPrice is handled inside the encoder; it is NOT a feature column.
cv_feature_columns = [c for c in dataset_train_input.columns if c not in ["Time", "DKPrice"]]
print(f"CV feature columns ({len(cv_feature_columns)}): {cv_feature_columns}")

start_time = time()
results = []
for comb_number, combination in enumerate(all_combinations, start=1):
    params = dict(zip(param_names, combination))
    if comb_number < start_combination:
        continue
    if int(params["layers"]) == 1 and float(params["dropout"]) > 0.0:
        continue

    WANDB_RUN_BASENAME = f"LSTM_AE_" + \
                    f"esize{params['encoder_hidden_size']}" + \
                    f"_elayers{params['layers']}" + \
                    f"_dsize{params['decoder_hidden_size']}" + \
                    f"_bs{params['batch_size']}" + \
                    f"_seqlen{params['sequence_length']}" + \
                    f"_lr{params['learning_rate']}" + \
                    f"_drop{params['dropout']}"

    print(f"\nCombination {comb_number}/{num_combinations}: {params}")
    print(
        f"Time: {(time() - start_time)/60:.2f} minutes - estimated total time: "
        f"{(time() - start_time)/comb_number*num_combinations/60:.2f} minutes"
    )
    run_name = (f"{WANDB_RUN_BASENAME}_comb{comb_number:03d}")

    run = wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config={
            "price_zone": PRICE_ZONE,
            "train_window": TRAIN_WINDOW,
            "val_window": VAL_WINDOW,
            "val_start": VAL_START,
            "predict_period": PREDICT_PERIOD,
            "stride": STRIDE,
            "split_setup": split_setup,
            "combination": int(comb_number),
            "num_combinations": int(num_combinations),
            "include_remaining_2024_in_prepared_train_data": bool(INCLUDE_REMAINING_2024_DURING_TRAINING),
            # pretrained encoder provenance
            "stage1_latent_dim":   int(best_stage1_params["latent_dim"]),
            "stage1_dense_layers": int(best_stage1_params["dense_layers"]),
            "stage1_recon_smape":  float(best_stage1_val_smape),
            "encoder_frozen":      True,
            # sub-grid membership for easy W&B grouping
            "fe_latent_dim":        int(best_stage1_params["latent_dim"]),
            "fe_dense_layers":      int(best_stage1_params["dense_layers"]),
            "enc_hidden_size":      int(params["encoder_hidden_size"]),
            "enc_layers":           int(params["layers"]),
            "dec_hidden_size":      int(params["decoder_hidden_size"]),
            **params,
        },
        tags=["lstm-ae", "hyperparameter-search", "cross-validation", "early-stopping",
              "pretrained-encoder", "frozen-encoder"],
        reinit=True,
        settings=wandb.Settings(start_method="thread"),
    )

    try:
        max_epochs = int(params["max_epochs"])
        patience = int(params["patience"])

        model = TorchLSTMAERegressor(
            # Use the latent_dim / dense_layers that match the pretrained encoder shape.
            # The grid may vary these, but the actual weights come from Stage 1.
            latent_dim=int(best_stage1_params["latent_dim"]),
            dense_layers=int(best_stage1_params["dense_layers"]),
            encoder_hidden_size=int(params["encoder_hidden_size"]),
            decoder_hidden_size=int(params["decoder_hidden_size"]),
            layers=int(params["layers"]),
            learning_rate=float(params["learning_rate"]),
            epochs=1,
            batch_size=int(params["batch_size"]),
            sequence_length=int(params["sequence_length"]),
            dropout=float(params["dropout"]),
            random_state=42,
            warm_start=True,
        )

        # Load the pretrained encoder weights from Stage 1 and freeze them.
        # _initialize_model_state() (called on the first fit()) will inject
        # these weights into LSTMAutoencoder.feature_encoder and exclude those
        # parameters from the optimizer.
        model.set_pretrained_encoder(best_stage1_encoder_state_dict, freeze=True)

        best_val_smape = float("inf")
        best_epoch = 0
        patience_counter = 0
        best_combination_results = None

        print(f"INCLUDE_REMAINING_2024_DURING_TRAINING: {INCLUDE_REMAINING_2024_DURING_TRAINING}")
        print(f"Number of input features (decoder): {len(cv_feature_columns)}")
        print(f"All input columns: {cv_feature_columns}")

        for epoch in range(1, max_epochs + 1):
            print(f"  Epoch {epoch}/{max_epochs}")
            combination_results = run_cross_validation(
                model=model,
                dataset_train=dataset_train,
                dataset_validation=dataset_validation,
                dataset_context=dataset_context,
                feature_columns=cv_feature_columns,
                include_remaining_2024=INCLUDE_REMAINING_2024_DURING_TRAINING,
                dk_zone=PRICE_ZONE,
                split_setup=split_setup,
                train_window=TRAIN_WINDOW,
                val_window=VAL_WINDOW,
                val_start=VAL_START,
                predict_period=PREDICT_PERIOD,
                stride=STRIDE,
                use_scaler=True,
                print_fold_results=False,
                plot=False,
                rf_models=rf_models,
                use_precomputed_feature_values=use_precomputed_feature_values,
                precomputed_feature_predictions=feature_predictions,
                use_forecasted_history=USE_FORECASTED_HISTORY,
            )

            val_smape = float(combination_results["overall_avg_weekly_smape"])

            if val_smape < best_val_smape:
                best_val_smape = val_smape
                best_epoch = epoch
                best_combination_results = combination_results
                patience_counter = 0
            else:
                patience_counter += 1

            wandb.log({
                "combination": int(comb_number),
                "epoch": int(epoch),
                "train_window": int(TRAIN_WINDOW),
                "val_window": int(VAL_WINDOW),
                "predict_period": int(PREDICT_PERIOD),
                # sub-grid params logged explicitly for W&B grouping/filtering
                "fe_latent_dim": int(best_stage1_params["latent_dim"]),
                "fe_dense_layers": int(best_stage1_params["dense_layers"]),
                "enc_hidden_size":        int(params["encoder_hidden_size"]),
                "enc_layers":             int(params["layers"]),
                "dec_hidden_size":        int(params["decoder_hidden_size"]),
                "learning_rate":          float(params["learning_rate"]),
                "batch_size":             int(params["batch_size"]),
                "sequence_length":        int(params["sequence_length"]),
                "max_epochs":             int(params["max_epochs"]),
                "patience":               int(params["patience"]),
                "val_SMAPE":              float(val_smape),
                "best_val_SMAPE":         float(best_val_smape),
                "patience_counter":       int(patience_counter),
            })

            if patience_counter >= patience:
                print("  Early stopping triggered.")
                break

        print(f"\n  best_val_SMAPE={best_val_smape:.3f}")

        if best_combination_results is None:
            raise RuntimeError("No validation results were produced for this combination.")

        row = {
            # sub-grid columns first for easy reading in the CSV
            "fe_latent_dim":          int(best_stage1_params["latent_dim"]),
            "fe_dense_layers":        int(best_stage1_params["dense_layers"]),
            "enc_hidden_size":        int(params["encoder_hidden_size"]),
            "enc_layers":             int(params["layers"]),
            "dec_hidden_size":        int(params["decoder_hidden_size"]),
            **params,
            "best_epoch": int(best_epoch),
            "epochs_trained": int(epoch),
            "price_zone": PRICE_ZONE,
            "train_window": str(TRAIN_WINDOW // 8760) + " years",
            "val_start": VAL_START.split(" ")[0],
            "avg_smape": best_val_smape,
            "avg_weekly_rmse": best_combination_results["overall_avg_weekly_rmse"],
            "avg_weekly_mae": best_combination_results["overall_avg_weekly_mae"],
            "avg_weekly_smape": best_combination_results["overall_avg_weekly_smape"],
            "avg_daily_rmse": best_combination_results["overall_avg_daily_rmse"],
            "avg_daily_mae": best_combination_results["overall_avg_daily_mae"],
            "avg_daily_smape": best_combination_results["overall_avg_daily_smape"],
            "avg_smape_day_1": best_combination_results["avg_smape_day_1"],
            "avg_smape_day_2": best_combination_results["avg_smape_day_2"],
            "avg_smape_day_3": best_combination_results["avg_smape_day_3"],
            "avg_smape_day_4": best_combination_results["avg_smape_day_4"],
            "avg_smape_day_5": best_combination_results["avg_smape_day_5"],
            "avg_smape_day_6": best_combination_results["avg_smape_day_6"],
            "avg_smape_day_7": best_combination_results["avg_smape_day_7"],
        }
        results.append(row)

        run.summary.update({
            "best_epoch": int(best_epoch),
            "best_val_smape": float(best_val_smape),
            "epochs_trained": int(epoch),
        })
    finally:
        wandb.finish()

results_df = pd.DataFrame(results).sort_values("avg_smape")

project_root_path = Path(project_root)
output_folder = project_root_path / "Deep learners" / "LSTM Autoencoder"
output_folder.mkdir(parents=True, exist_ok=True)

base_filename = f"{PRICE_ZONE}_lstm_ae_search_results"
filename = output_folder / f"{base_filename}.csv"
counter = 1
while filename.exists():
    filename = output_folder / f"{base_filename}_{counter}.csv"
    counter += 1

results_df.to_csv(filename, index=False, decimal=",")
print(f"\nResults saved to: {filename}")
display(results_df.head(10))


Using pretrained encoder from Stage 1:
  latent_dim   = 48
  dense_layers = 1
  best recon SMAPE = 23.7420

Total combinations: 144
CV feature columns (32): ['OffshoreWindPower', 'OnshoreWindPower', 'HydroPower', 'SolarPower', 'Biomass', 'Biogas', 'Waste', 'FossilGas', 'FossilOil', 'FossilHardCoal', 'ExchangeGreatBelt', 'ExchangeGermany', 'ExchangeSweden', 'ExchangeNorway', 'ExchangeNetherlands', 'WindSpeed', 'Radiation', 'DEPrice', 'NO2Price', 'SE3Price', 'SE4Price', 'NLPrice', 'GrossCon', 'TotalProduction', 'Year', 'Month', 'Day', 'WeekDay', 'Hour', 'OffshoreWindCapacity', 'OnshoreWindCapacity', 'SolarPowerCapacity']

Combination 1/144: {'encoder_hidden_size': 16, 'layers': 1, 'decoder_hidden_size': 16, 'learning_rate': 0.001, 'max_epochs': 60, 'patience': 10, 'batch_size': 32, 'sequence_length': 24, 'dropout': 0.0}
Time: 0.00 minutes - estimated total time: 0.00 minutes


INCLUDE_REMAINING_2024_DURING_TRAINING: True
Number of input features (decoder): 32
All input columns: ['OffshoreWindPower', 'OnshoreWindPower', 'HydroPower', 'SolarPower', 'Biomass', 'Biogas', 'Waste', 'FossilGas', 'FossilOil', 'FossilHardCoal', 'ExchangeGreatBelt', 'ExchangeGermany', 'ExchangeSweden', 'ExchangeNorway', 'ExchangeNetherlands', 'WindSpeed', 'Radiation', 'DEPrice', 'NO2Price', 'SE3Price', 'SE4Price', 'NLPrice', 'GrossCon', 'TotalProduction', 'Year', 'Month', 'Day', 'WeekDay', 'Hour', 'OffshoreWindCapacity', 'OnshoreWindCapacity', 'SolarPowerCapacity']
  Epoch 1/60
Model trained in 2.25s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 179.872
  Epoch 2/60
Model trained in 2.18s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 169.774
  Epoch 3/60
Model trained in 2.12s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 161.420
  Epoch 4/60
Model trained in 2.25s. Now validating on 4 folds...



batch_size,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_SMAPE,█▇▇▆▆▅▅▅▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
combination,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
dec_hidden_size,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
enc_hidden_size,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
enc_layers,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
fe_dense_layers,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
fe_latent_dim,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+8,...



Combination 3/144: {'encoder_hidden_size': 16, 'layers': 1, 'decoder_hidden_size': 16, 'learning_rate': 0.001, 'max_epochs': 60, 'patience': 10, 'batch_size': 32, 'sequence_length': 168, 'dropout': 0.0}
Time: 2.04 minutes - estimated total time: 97.97 minutes


INCLUDE_REMAINING_2024_DURING_TRAINING: True
Number of input features (decoder): 32
All input columns: ['OffshoreWindPower', 'OnshoreWindPower', 'HydroPower', 'SolarPower', 'Biomass', 'Biogas', 'Waste', 'FossilGas', 'FossilOil', 'FossilHardCoal', 'ExchangeGreatBelt', 'ExchangeGermany', 'ExchangeSweden', 'ExchangeNorway', 'ExchangeNetherlands', 'WindSpeed', 'Radiation', 'DEPrice', 'NO2Price', 'SE3Price', 'SE4Price', 'NLPrice', 'GrossCon', 'TotalProduction', 'Year', 'Month', 'Day', 'WeekDay', 'Hour', 'OffshoreWindCapacity', 'OnshoreWindCapacity', 'SolarPowerCapacity']
  Epoch 1/60
Model trained in 2.44s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 179.881
  Epoch 2/60
Model trained in 2.32s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 169.853
  Epoch 3/60
Model trained in 2.24s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 161.517
  Epoch 4/60
Model trained in 2.26s. Now validating on 4 folds...



...


Combination 144/144: {'encoder_hidden_size': 48, 'layers': 2, 'decoder_hidden_size': 48, 'learning_rate': 0.001, 'max_epochs': 60, 'patience': 10, 'batch_size': 64, 'sequence_length': 168, 'dropout': 0.2}
Time: 199.16 minutes - estimated total time: 199.16 minutes


INCLUDE_REMAINING_2024_DURING_TRAINING: True
Number of input features (decoder): 32
All input columns: ['OffshoreWindPower', 'OnshoreWindPower', 'HydroPower', 'SolarPower', 'Biomass', 'Biogas', 'Waste', 'FossilGas', 'FossilOil', 'FossilHardCoal', 'ExchangeGreatBelt', 'ExchangeGermany', 'ExchangeSweden', 'ExchangeNorway', 'ExchangeNetherlands', 'WindSpeed', 'Radiation', 'DEPrice', 'NO2Price', 'SE3Price', 'SE4Price', 'NLPrice', 'GrossCon', 'TotalProduction', 'Year', 'Month', 'Day', 'WeekDay', 'Hour', 'OffshoreWindCapacity', 'OnshoreWindCapacity', 'SolarPowerCapacity']
  Epoch 1/60
Model trained in 1.98s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 173.804
  Epoch 2/60
Model trained in 1.79s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 161.095
  Epoch 3/60
Model trained in 1.78s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 150.781
  Epoch 4/60
Model trained in 1.74s. Now validating on 4 folds...



batch_size,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_SMAPE,█▇▆▆▅▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
combination,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
dec_hidden_size,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
enc_hidden_size,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
enc_layers,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
fe_dense_layers,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
fe_latent_dim,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+8,...



Results saved to: c:\Users\n_and\OneDrive\Delt skrivebord\Data Science\Speciale\Energinet\Delte scripts\Speciale_Kode\Deep learners\LSTM Autoencoder\DK1_lstm_ae_search_results_2.csv


,fe_latent_dim,fe_dense_layers,enc_hidden_size,enc_layers,dec_hidden_size,encoder_hidden_size,layers,decoder_hidden_size,learning_rate,max_epochs,...,avg_daily_rmse,avg_daily_mae,avg_daily_smape,avg_smape_day_1,avg_smape_day_2,avg_smape_day_3,avg_smape_day_4,avg_smape_day_5,avg_smape_day_6,avg_smape_day_7
44,48,1,32,1,48,32,1,48,0.001,60,...,235.706736,203.647877,64.966176,58.773976,41.805386,46.737987,57.492625,66.027041,83.719662,100.206553
45,48,1,32,1,48,32,1,48,0.001,60,...,229.954432,196.254736,65.131135,64.200804,42.458226,38.830595,54.601942,66.167897,86.010769,103.647708
4,48,1,16,1,32,16,1,32,0.001,60,...,232.648581,204.880818,65.796253,60.830447,41.708680,36.926446,57.400812,69.341050,85.720694,108.645641
23,48,1,16,2,32,16,2,32,0.001,60,...,227.850522,199.247940,66.541318,61.313756,44.000359,38.808087,56.705003,71.003377,87.331083,106.627564
91,48,1,48,2,16,48,2,16,0.001,60,...,240.523340,207.304739,66.904494,60.186715,44.403284,55.375297,64.452800,65.666230,84.209903,94.037232
20,48,1,16,2,32,16,2,32,0.001,60,...,238.849481,208.165635,67.015614,61.212212,46.963017,39.640465,55.621166,72.732356,81.802412,111.137670
10,48,1,16,1,48,16,1,48,0.001,60,...,240.159701,208.170218,67.142702,67.237162,44.022243,43.280125,58.402689,65.656897,85.938457,105.461338
1,48,1,16,1,16,16,1,16,0.001,60,...,240.389645,206.697420,67.461661,57.553599,45.667028,49.081439,61.699088,67.949042,82.698480,107.582954
11,48,1,16,1,48,16,1,48,0.001,60,...,239.433466,206.881508,67.927568,64.487616,45.300430,43.162961,58.943477,69.109259,88.258781,106.230453
95,48,1,48,2,32,48,2,32,0.001,60,...,246.501370,217.882322,68.275797,70.018507,51.446428,39.376910,55.117551,66.270401,85.745081,109.955705


## Train final model

In [ ]:

import copy
import numpy as np
import pandas as pd
from pathlib import Path
import wandb
import tempfile
import joblib
from sklearn.preprocessing import StandardScaler
from Modules.Validation3_AE import run_cross_validation

# =====================================================================================
# Stage 3 – Train final LSTM Autoencoder
#
# Parameters are taken from the best Stage 2 row (results_df).
# The pretrained feature encoder from Stage 1 is loaded UNFROZEN so all weights
# can fine-tune together end-to-end.
# =====================================================================================

# ---------------------------------------------------------------------------
# Validate prerequisites
# ---------------------------------------------------------------------------
required_globals = [
    "dataset_train", "dataset_train_input", "dataset_validation",
    "dataset_context", "INCLUDE_REMAINING_2024_DURING_TRAINING",
    "INCLUDE_LAGS", "USE_FORECASTED_HISTORY",
    "results_df",                          # Stage 2 search results
    "best_stage1_encoder_state_dict",      # pretrained encoder weights from Stage 1
    "best_stage1_params",                  # Stage 1 best latent_dim / dense_layers
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise ValueError(
        f"Missing prerequisites – run Stage 1 and Stage 2 first. "
        f"Missing: {missing_globals}"
    )

if dataset_train.empty:
    raise ValueError("Prepared dataset_train is empty; cannot train final model.")
if dataset_validation.empty:
    raise ValueError("dataset_validation is empty; cannot run final training with early stopping.")
if best_stage1_encoder_state_dict is None:
    raise ValueError("best_stage1_encoder_state_dict is None. Re-run Stage 1.")

# ---------------------------------------------------------------------------
# Derive parameters from the best Stage 2 row
# ---------------------------------------------------------------------------
best_stage2_row = results_df.iloc[0]

params = {
    # Feature encoder shape – fixed from Stage 1
    "latent_dim":            int(best_stage1_params["latent_dim"]),
    "dense_layers":          int(best_stage1_params["dense_layers"]),
    # LSTM / training – from Stage 2 best
    "encoder_hidden_size":   int(best_stage2_row["encoder_hidden_size"]),
    "decoder_hidden_size":   int(best_stage2_row["decoder_hidden_size"]),
    "layers":                int(best_stage2_row["layers"]),
    "learning_rate":         float(best_stage2_row["learning_rate"]),
    "batch_size":            int(best_stage2_row["batch_size"]),
    "sequence_length":       int(best_stage2_row["sequence_length"]),
    "dropout":               float(best_stage2_row["dropout"]),
}

print("=== Stage 3 parameters (derived from Stage 1 + Stage 2 best) ===")
for k, v in params.items():
    print(f"  {k}: {v}")
print(f"  (Stage 2 best avg_smape: {float(best_stage2_row['avg_smape']):.4f})")

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
PREDICT_PERIOD = 4 * 168
MAX_EPOCHS = 80
PATIENCE = 30
MIN_DELTA = 0.0
WANDB_PROJECT = "LSTM_AE_final"
WANDB_RUN_NAME = (
    f"{PRICE_ZONE}_LSTM_AE_{TRAIN_WINDOW//8760}y_"
    f"2024{'incl' if INCLUDE_REMAINING_2024_DURING_TRAINING else 'excl'}"
    f"_DKPriceLag1{'_incl' if INCLUDE_PRICE_LAG1_AS_INPUT else '_excl'}"
    f"_lags{'_incl' if INCLUDE_LAGS else '_excl'}"
    f"_{PREDICT_PERIOD//168}val"
)
save_model_to_wandb = True
save_model_to_disk = False

feature_columns = [c for c in dataset_train_input.columns if c not in ["Time", "DKPrice"]]
if not feature_columns:
    raise ValueError("No feature columns found in dataset_train_input.")

output_root = Path(project_root) / "Deep learners" / "LSTM Autoencoder"
output_root.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# W&B run
# ---------------------------------------------------------------------------
run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "price_zone": PRICE_ZONE,
        "train_window": int(TRAIN_WINDOW),
        "training_rows": int(len(dataset_train)),
        "validation_rows": int(len(dataset_validation)),
        "train_start_time": str(dataset_train["Time"].min()),
        "train_end_time": str(dataset_train["Time"].max()),
        "val_start": VAL_START,
        "val_window": int(VAL_WINDOW),
        "predict_period": int(PREDICT_PERIOD),
        "stride": int(STRIDE),
        "include_remaining_2024_in_prepared_train_data": bool(INCLUDE_REMAINING_2024_DURING_TRAINING),
        "include_lags": bool(INCLUDE_LAGS),
        "use_forecasted_history": bool(USE_FORECASTED_HISTORY),
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "min_delta": MIN_DELTA,
        "save_model_to_wandb": bool(save_model_to_wandb),
        "decoder_horizon": 168,
        "stage1_recon_smape": float(best_stage1_val_smape),
        "stage2_best_val_smape": float(best_stage2_row["avg_smape"]),
        "encoder_pretrained": True,
        "encoder_frozen": False,
        **params,
    },
    tags=["lstm-ae", "final-model", "early-stopping", "pretrained-encoder", "unfrozen-encoder"],
    reinit=True,
    settings=wandb.Settings(start_method="thread"),
)

print(f"\nTraining rows: {len(dataset_train)}")
print(f"Training window: {dataset_train['Time'].min()} -> {dataset_train['Time'].max()}")
print(f"Validation rows: {len(dataset_validation)}")
print(f"Include remainder_2024_for_train: {INCLUDE_REMAINING_2024_DURING_TRAINING}")
print(f"Include lag features: {INCLUDE_LAGS}")
print(f"Use forecasted history: {USE_FORECASTED_HISTORY}")
print(f"Feature columns (decoder, {len(feature_columns)}): {feature_columns}")

# ---------------------------------------------------------------------------
# Build model – load pretrained encoder UNFROZEN (freeze=False) so all weights
# fine-tune end-to-end during Stage 3.
# ---------------------------------------------------------------------------
model = TorchLSTMAERegressor(
    latent_dim=params["latent_dim"],
    encoder_hidden_size=params["encoder_hidden_size"],
    decoder_hidden_size=params["decoder_hidden_size"],
    layers=params["layers"],
    dense_layers=params["dense_layers"],
    learning_rate=params["learning_rate"],
    epochs=1,
    batch_size=params["batch_size"],
    sequence_length=params["sequence_length"],
    dropout=params["dropout"],
    random_state=42,
    log_epoch_metrics=True,
    log_prefix="final_",
    warm_start=True,
)

# Load pretrained encoder weights but keep them trainable (freeze=False)
model.set_pretrained_encoder(best_stage1_encoder_state_dict, freeze=False)

# ---------------------------------------------------------------------------
# Training loop with early stopping
# ---------------------------------------------------------------------------
best_val_smape = float("inf")
best_epoch = 0
patience_counter = 0
best_model = None
epoch_history = []

for epoch in range(1, MAX_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{MAX_EPOCHS}")

    epoch_results = run_cross_validation(
        model=model,
        dataset_train=dataset_train,
        dataset_validation=dataset_validation,
        dataset_context=dataset_context,
        feature_columns=feature_columns,
        include_remaining_2024=INCLUDE_REMAINING_2024_DURING_TRAINING,
        dk_zone=PRICE_ZONE,
        split_setup=2,
        train_window=TRAIN_WINDOW,
        val_window=VAL_WINDOW,
        val_start=VAL_START,
        predict_period=PREDICT_PERIOD,
        stride=STRIDE,
        use_scaler=True,
        print_fold_results=False,
        plot=False,
        rf_models=rf_models,
        use_precomputed_feature_values=use_precomputed_feature_values,
        precomputed_feature_predictions=feature_predictions,
        use_forecasted_history=USE_FORECASTED_HISTORY,
    )

    train_mse = float(model.epoch_losses_[-1]) if hasattr(model, "epoch_losses_") else float("nan")
    train_smape = float(model.epoch_smapes_[-1]) if hasattr(model, "epoch_smapes_") else float("nan")
    val_smape = float(epoch_results["overall_avg_weekly_smape"])

    improved = val_smape < (best_val_smape - MIN_DELTA)
    if improved:
        best_val_smape = val_smape
        best_epoch = epoch
        patience_counter = 0
        best_model = copy.deepcopy(model)
        print(f"  Validation SMAPE improved: {val_smape:.4f} (best so far)")
    else:
        patience_counter += 1
        print(f"  Validation SMAPE: {val_smape:.4f} (patience {patience_counter}/{PATIENCE})")

    epoch_row = {
        "epoch": epoch,
        "train_MSE": train_mse,
        "train_SMAPE": train_smape,
        "val_SMAPE": val_smape,
        "best_val_SMAPE": best_val_smape,
        "patience_counter": int(patience_counter),
    }
    epoch_history.append(epoch_row)
    wandb.log(epoch_row)

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}. Best epoch: {best_epoch}.")
        break

if best_epoch == 0:
    raise RuntimeError("No valid epoch found during final training with early stopping.")

print(f"\n=== Stage 3 Training Complete ===")
print(f"Best epoch: {best_epoch}")
print(f"Best validation SMAPE: {best_val_smape:.4f}")

final_model = best_model

epoch_metrics_df = pd.DataFrame(epoch_history)
wandb.log({"final_epoch_metrics": wandb.Table(dataframe=epoch_metrics_df)})
wandb.log({
    "best_epoch": int(best_epoch),
    "best_val_smape": float(best_val_smape),
})
run.summary.update({
    "best_epoch": int(best_epoch),
    "best_val_smape": float(best_val_smape),
    "epochs_trained": int(epoch),
})

if save_model_to_wandb:
    model_artifact = wandb.Artifact(name=f"{WANDB_RUN_NAME}_model", type="model")
    with tempfile.TemporaryDirectory() as tmpdir:
        model_path = Path(tmpdir) / f"{WANDB_RUN_NAME}_model.joblib"
        joblib.dump(final_model, model_path, compress=3)
        model_artifact.add_file(str(model_path), name="model.joblib")
        run.log_artifact(model_artifact)
    print("Model stored in W&B artifact.")
else:
    print("Model not saved to W&B. Set save_model_to_wandb = True to upload it.")

print(f"Trained final LSTM AE on the prepared {PRICE_ZONE} train set (best epoch: {best_epoch}).")

wandb.finish()


=== Stage 3 parameters (derived from Stage 1 + Stage 2 best) ===
  latent_dim: 48
  dense_layers: 1
  encoder_hidden_size: 16
  decoder_hidden_size: 32
  layers: 2
  learning_rate: 0.001
  batch_size: 64
  sequence_length: 24
  dropout: 0.2
  (Stage 2 best avg_smape: 70.9189)



Training rows: 22944
Training window: 2022-01-01 00:00:00 -> 2024-12-31 23:00:00
Validation rows: 2688
Include remainder_2024_for_train: True
Include lag features: False
Use forecasted history: True
Feature columns (decoder, 33): ['OffshoreWindPower', 'OnshoreWindPower', 'HydroPower', 'SolarPower', 'Biomass', 'Biogas', 'Waste', 'FossilGas', 'FossilOil', 'FossilHardCoal', 'ExchangeGreatBelt', 'ExchangeGermany', 'ExchangeSweden', 'ExchangeNorway', 'ExchangeNetherlands', 'WindSpeed', 'Radiation', 'DEPrice', 'NO2Price', 'SE3Price', 'SE4Price', 'NLPrice', 'GrossCon', 'TotalProduction', 'Year', 'Month', 'Day', 'WeekDay', 'Hour', 'OffshoreWindCapacity', 'OnshoreWindCapacity', 'SolarPowerCapacity', 'DKPrice_lag1']

Epoch 1/80
Model trained in 1.86s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 182.245
  Validation SMAPE improved: 182.2450 (best so far)

Epoch 2/80


Model trained in 1.83s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 173.020
  Validation SMAPE improved: 173.0198 (best so far)

Epoch 3/80
Model trained in 1.89s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 164.970
  Validation SMAPE improved: 164.9697 (best so far)

Epoch 4/80
Model trained in 1.81s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 157.841
  Validation SMAPE improved: 157.8415 (best so far)

Epoch 5/80
Model trained in 1.86s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 151.295
  Validation SMAPE improved: 151.2952 (best so far)

Epoch 6/80
Model trained in 1.78s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 145.199
  Validation SMAPE improved: 145.1988 (best so far)

Epoch 7/80
Model trained in 1.82s. Now validating on 4 folds...

Average SMAPE across all weeks in all folds: 139.537
  Validation SMAPE improved: 139.5370 

best_epoch,▁
best_val_SMAPE,█▇▇▆▅▅▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_smape,▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
final_epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇███
final_train_MSE_loss,███▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
final_train_mae,███▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
final_train_rmse,████▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁
final_train_smape,██▇▇▇▆▆▆▅▅▅▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
patience_counter,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██
+3,...


Test Final Model

In [ ]:
# Fit scaler on the same training tail used during training.
train_for_scaler = history_source.sort_values("Time").copy()
train_for_scaler = train_for_scaler.tail(TRAIN_WINDOW).reset_index(drop=True)
# DKPrice_lag1 is stored as Price_lag1 in raw source DataFrames; rename to match feature_columns.
if "DKPrice_lag1" not in train_for_scaler.columns and "Price_lag1" in train_for_scaler.columns:
    train_for_scaler["DKPrice_lag1"] = train_for_scaler["Price_lag1"]
X_scaler = train_for_scaler[feature_columns].astype(np.float32)
scaler = StandardScaler()
scaler.fit(X_scaler)

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.
wandb: Currently logged in as: nande24 (Energinet_speciale) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb:   1 of 1 files downloaded.  


Loaded model artifact: DK1_LSTM_AE_2y_2024incl_DKPriceLag1_incl_lags_excl_4val_model:latest
Test window: 2025-01-01 00:00:00 -> 2025-12-31 23:00:00 (8760 rows)
Feature columns (decoder, 33): ['OffshoreWindPower', 'OnshoreWindPower', 'HydroPower', 'SolarPower', 'Biomass', 'Biogas', 'Waste', 'FossilGas', 'FossilOil', 'FossilHardCoal', 'ExchangeGreatBelt', 'ExchangeGermany', 'ExchangeSweden', 'ExchangeNorway', 'ExchangeNetherlands', 'WindSpeed', 'Radiation', 'DEPrice', 'NO2Price', 'SE3Price', 'SE4Price', 'NLPrice', 'GrossCon', 'TotalProduction', 'Year', 'Month', 'Day', 'WeekDay', 'Hour', 'OffshoreWindCapacity', 'OnshoreWindCapacity', 'SolarPowerCapacity', 'DKPrice_lag1']


KeyError: "['DKPrice_lag1'] not in index"

Shap analysis

In [ ]:

import gc
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import shap
import wandb

# ==========================
# SHAP configuration (LSTM AE)
# Note: predict() uses a constant-feature decoder with zero encoder baseline.
# SHAP values reflect sensitivity of the mean predicted price to each decoder
# feature value held constant across the 168-hour horizon.
# ==========================
quick_mode = False
eval_size = 300            # evaluation sample size
bg_size = 150              # background sample size for KernelExplainer
chunk_size = 25            # lower if memory/runtime is high
nsamples = 100             # SHAP Monte Carlo samples per explained point
include_beeswarm = True
include_waterfall = True

PRICE_ZONE = globals().get("PRICE_ZONE", "DK1")
WANDB_PROJECT = globals().get("WANDB_PROJECT", "LSTM_AE")
WANDB_RUN_NAME = WANDB_RUN_NAME  # Must match the final training run name from Stage 3
WANDB_RUN_NAME = "DK1_LSTM_AE_2y_2024incl_DKPriceLag1_excl_lags_excl_4val"
WANDB_ARTIFACT_NAME = f"{WANDB_RUN_NAME}_model"
WANDB_SHAP_RUN_NAME = f"{PRICE_ZONE}_lstm_ae_shap"


# Use dataset_train_input (feature columns only, no DKPrice) as the SHAP background/eval set.
if "dataset_train_input" not in globals() or dataset_train_input is None:
    raise ValueError("Missing dataset_train_input. Run the data loading cell first.")

feature_columns = [c for c in dataset_train_input.columns if c not in ["Time", "DKPrice"]]
if not feature_columns:
    raise ValueError("No feature columns found in dataset_train_input after removing ['Time', 'DKPrice'].")

X_train_shap = dataset_train_input.loc[:, feature_columns].copy()
if len(X_train_shap) == 0:
    raise ValueError("No rows found in dataset_train_input for SHAP analysis.")

X_train_shap = X_train_shap.astype(np.float32, copy=False)

# Start a dedicated W&B run for SHAP and load latest model artifact
wandb_run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_SHAP_RUN_NAME,
    job_type="shap-analysis",
    config={
        "price_zone": PRICE_ZONE,
        "train_hours": int(TRAIN_WINDOW),
        "artifact_name": WANDB_ARTIFACT_NAME,
        "eval_size_requested": int(eval_size),
        "bg_size_requested": int(bg_size),
        "nsamples": int(nsamples),
    },
    reinit=True,
    settings=wandb.Settings(start_method="thread"),
)

model_artifact = wandb_run.use_artifact(f"{WANDB_ARTIFACT_NAME}:latest")
artifact_dir = Path(model_artifact.download())
model_path = artifact_dir / "model.joblib"
if not model_path.exists():
    raise ValueError(f"Could not find model.joblib in downloaded artifact: {artifact_dir}")

model = joblib.load(model_path)
print(f"Loaded LSTM AE model artifact: {WANDB_ARTIFACT_NAME}:latest")

mem = psutil.virtual_memory()
print(f"Available RAM before SHAP: {mem.available / (1024**3):.2f} GB")
print(f"Training set size for SHAP: {len(X_train_shap)} samples")
print(f"Number of features: {len(feature_columns)}")

if quick_mode:
    bg_size = min(30, len(X_train_shap))
    eval_size = min(60, len(X_train_shap))
    chunk_size = 10
    nsamples = 50
else:
    bg_size = min(bg_size, len(X_train_shap))
    eval_size = min(eval_size, len(X_train_shap))
    chunk_size = max(1, min(chunk_size, eval_size))

X_bg = shap.sample(X_train_shap, bg_size, random_state=42)
X_eval = shap.sample(X_train_shap, eval_size, random_state=42)

print(f"\nBackground sample size (X_bg): {len(X_bg)}")
print(f"Evaluation sample size (X_eval): {len(X_eval)}")
print(f"Chunk size: {chunk_size}")
print(f"Kernel SHAP nsamples: {nsamples}")

def predict_fn(x):
    x_df = pd.DataFrame(x, columns=feature_columns)
    preds = model.predict(x_df)
    return np.asarray(preds).reshape(-1)

explainer = shap.KernelExplainer(predict_fn, X_bg.values)

# Compute SHAP values in chunks and print progress
shap_chunks = []
n_chunks = (len(X_eval) + chunk_size - 1) // chunk_size
for idx, start in enumerate(range(0, len(X_eval), chunk_size), start=1):
    stop = min(start + chunk_size, len(X_eval))
    print(f"Computing SHAP chunk {idx}/{n_chunks} (rows {start}:{stop})...", flush=True)
    X_chunk = X_eval.iloc[start:stop]
    shap_chunk = explainer.shap_values(X_chunk.values, nsamples=nsamples)
    shap_chunks.append(np.asarray(shap_chunk))

shap_values = np.vstack(shap_chunks)
gc.collect()

print("\nSHAP analysis complete.")

# Global importance bar plot
plt.figure(figsize=(12, 6))
shap.summary_plot(shap_values, X_eval, plot_type="bar", show=False)
plt.tight_layout()
wandb_run.log({"shap_bar": wandb.Image(plt.gcf())})
plt.show()
plt.close()

# Beeswarm plot
if include_beeswarm:
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_eval, show=False)
    plt.tight_layout()
    wandb_run.log({"shap_beeswarm": wandb.Image(plt.gcf())})
    plt.show()
    plt.close()

# Waterfall plot for first sample
if include_waterfall:
    i = 0
    base_value = float(np.mean(predict_fn(X_bg.values)))
    explanation = shap.Explanation(
        values=shap_values[i],
        base_values=base_value,
        data=X_eval.iloc[i].values,
        feature_names=X_eval.columns.tolist(),
    )
    plt.figure(figsize=(10, 4))
    shap.plots.waterfall(explanation, show=False)
    plt.tight_layout()
    wandb_run.log({"shap_waterfall": wandb.Image(plt.gcf())})
    plt.show()
    plt.close()

# Log mean absolute SHAP as a table
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": feature_columns,
    "mean_abs_shap": mean_abs_shap,
}).sort_values("mean_abs_shap", ascending=False)

wandb_run.log({"shap_importance_table": wandb.Table(dataframe=importance_df)})
wandb_run.summary.update({
    "shap_eval_size": int(len(X_eval)),
    "shap_bg_size": int(len(X_bg)),
    "top_feature": str(importance_df.iloc[0]["feature"]),
    "top_feature_mean_abs_shap": float(importance_df.iloc[0]["mean_abs_shap"]),
})

mem_after = psutil.virtual_memory()
print(f"Available RAM after SHAP cleanup: {mem_after.available / (1024**3):.2f} GB")

# Cleanup large objects explicitly
del X_bg, X_eval, X_train_shap, shap_chunks, shap_values
gc.collect()

wandb.finish()


wandb:   1 of 1 files downloaded.  
c:\Users\n_and\OneDrive\Delt skrivebord\Data Science\Speciale\Energinet\py_3.10_blackwell\lib\site-packages\torch\nn\modules\rnn.py:1141: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1480.)
  result = _VF.lstm(
Using 150 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


Loaded LSTM AE model artifact: DK1_LSTM_AE_2y_2024incl_DKPriceLag1_excl_lags_excl_4val_model:latest
Available RAM before SHAP: 18.54 GB
Training set size for SHAP: 22944 samples
Number of features: 32

Background sample size (X_bg): 150
Evaluation sample size (X_eval): 300
Chunk size: 25
Kernel SHAP nsamples: 100
Computing SHAP chunk 1/12 (rows 0:25)...


100%|██████████| 25/25 [05:52<00:00, 14.10s/it]

Computing SHAP chunk 2/12 (rows 25:50)...



100%|██████████| 25/25 [05:55<00:00, 14.22s/it]

...

Computing SHAP chunk 6/12 (rows 125:150)...



100%|██████████| 25/25 [05:50<00:00, 14.03s/it]

Computing SHAP chunk 7/12 (rows 150:175)...



 56%|█████▌    | 14/25 [03:24<02:40, 14.63s/it]


KeyboardInterrupt: 